# Notebook 1 — Dynamical Systems Analysis

**NeuroControl-PINN** | Physics-Informed Neural Networks for Nonlinear MPC

---

This notebook introduces the three nonlinear dynamical systems used in this project:

| System | States | Difficulty | Application |
|--------|--------|-----------|-------------|
| **Van der Pol Oscillator** | 2 | Medium | Limit cycle control, power electronics |
| **Inverted Pendulum (Cart-Pole)** | 4 | Hard | Robotics, aerospace |
| **CSTR Reactor** | 2 | Hard | Chemical process control |

We will:
1. Simulate each system and visualise phase portraits
2. Analyse the nonlinear dynamics and equilibrium structure
3. Verify the simulation against analytical solutions where available

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from src.systems import VanDerPolSystem, CartPoleSystem, CSTRSystem

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## 1. Van der Pol Oscillator

The Van der Pol oscillator is described by:

$$\dot{x}_1 = x_2$$
$$\dot{x}_2 = \mu (1 - x_1^2) x_2 - x_1 + u$$

For $\mu > 0$, the unforced system ($u=0$) exhibits a **stable limit cycle**. Our goal is to use MPC + PINN to drive the state to the unstable equilibrium at the origin.

In [ ]:
vdp = VanDerPolSystem(mu=1.0)
print(vdp)
print(f"State labels:   {vdp.state_labels}")
print(f"Control labels: {vdp.control_labels}")
print(f"State bounds:   {vdp.state_bounds}")
print(f"Control bounds: {vdp.control_bounds}")

In [ ]:
# Simulate multiple trajectories (no control)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = plt.cm.tab10(np.linspace(0, 0.8, 6))

for i, x0 in enumerate([[0.1, 0.0], [2.5, 0.0], [-2.0, 1.0],
                         [0.5, 2.0], [-1.0, -1.5], [3.0, -0.5]]):
    data = vdp.simulate(
        x0=np.array(x0),
        u_traj=np.zeros((400, 1)),
        t_span=(0, 20),
        dt=0.05,
    )
    axes[0].plot(data['t'], data['x'][:, 0], color=colors[i], alpha=0.8)
    axes[1].plot(data['x'][:, 0], data['x'][:, 1], color=colors[i], alpha=0.8)
    axes[1].plot(x0[0], x0[1], 'o', color=colors[i], ms=5)

axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('x₁ (position)')
axes[0].set_title('Van der Pol — Time Series')
axes[0].grid(alpha=0.3)

axes[1].set_xlabel('x₁ (position)')
axes[1].set_ylabel('x₂ (velocity)')
axes[1].set_title('Van der Pol — Phase Portrait (μ=1)')
axes[1].plot(0, 0, 'k*', ms=12, label='Equilibrium', zorder=10)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Unforced Van der Pol Oscillator', fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Cart-Pole (Inverted Pendulum)

The 4-dimensional cart-pole system:

$$\mathbf{x} = [p,\; \dot{p},\; \theta,\; \dot{\theta}]^\top$$

The goal is to **balance the pole upright** ($\theta = 0$) while keeping the cart near $p = 0$.

This is a classic benchmark in nonlinear control — the linearised system around $\theta=0$ is unstable.

In [ ]:
cp = CartPoleSystem()

# Linearise around the upright equilibrium
x_eq = np.zeros(4)
u_eq = np.zeros(1)
A, B = cp.linearize(x_eq, u_eq)

print("Linearised A matrix:")
print(np.round(A, 3))
print("\nLinearised B matrix:")
print(np.round(B, 3))
print("\nEigenvalues of A (open-loop):")
eigs = np.linalg.eigvals(A)
print(eigs)
print(f"\nSystem is {'UNSTABLE' if np.any(np.real(eigs) > 0) else 'stable'} in open loop")

In [ ]:
# Free-fall from small perturbation (shows instability)
x0 = np.array([0.0, 0.0, 0.05, 0.0])  # 0.05 rad ≈ 3° tilt
data_free = cp.simulate(
    x0=x0,
    u_traj=np.zeros((200, 1)),
    t_span=(0, 4),
    dt=0.02,
)

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
labels = cp.state_labels

for i, ax in enumerate(axes.flat):
    ax.plot(data_free['t'], data_free['x'][:, i], '#E91E63', lw=2)
    ax.set_ylabel(labels[i])
    ax.grid(alpha=0.3)
    ax.set_xlabel('Time (s)')

plt.suptitle(f'Cart-Pole — Uncontrolled (x₀ = {x0})', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. CSTR — Exothermic Reactor

The CSTR exhibits **multiple steady states** due to the Arrhenius exponential in the reaction rate:

$$\dot{C}_A = \frac{q}{V}(C_{Af} - C_A) - k_0 e^{-E/RT} C_A$$
$$\dot{T} = \frac{q}{V}(T_f - T) + \frac{-\Delta H}{\rho C_p} k_0 e^{-E/RT} C_A + \frac{UA}{\rho C_p V}(T_c - T)$$

The control task is to drive the reactor from the **low-conversion** steady state to the **high-conversion** steady state.

In [ ]:
cstr = CSTRSystem()

print("CSTR Steady States at Tc = 300 K:")
for ss in cstr.steady_states:
    print(f"  {ss['label']:35s}  CA={ss['CA']:.4f} mol/L  T={ss['T']:.1f} K")

In [ ]:
# Phase portrait of the CSTR (CA vs T)
fig, ax = plt.subplots(figsize=(8, 6))

CA_grid = np.linspace(0.0, 1.0, 20)
T_grid = np.linspace(300, 450, 20)
CA_mesh, T_mesh = np.meshgrid(CA_grid, T_grid)

dCA = np.zeros_like(CA_mesh)
dT = np.zeros_like(T_mesh)

for i in range(20):
    for j in range(20):
        x = np.array([CA_mesh[i, j], T_mesh[i, j]])
        u = np.array([300.0])
        dxdt = cstr.dynamics(0, x, u)
        dCA[i, j] = dxdt[0]
        dT[i, j] = dxdt[1]

speed = np.sqrt(dCA**2 + dT**2)
ax.streamplot(CA_mesh, T_mesh, dCA, dT,
              color=speed / speed.max(), cmap='Oranges',
              linewidth=0.8, arrowsize=0.8, density=1.2)

# Mark steady states
for ss in cstr.steady_states:
    stable = 'stable' in ss['label']
    ax.plot(ss['CA'], ss['T'],
            'g*' if stable else 'r^',
            ms=14, zorder=10,
            label=ss['label'])

ax.set_xlabel('C_A (mol/L)')
ax.set_ylabel('T (K)')
ax.set_title('CSTR Phase Portrait (Tc = 300 K)', fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 4. Training Data Preview

Let's generate and inspect the training data that will be used to train the PINNs.

In [ ]:
from src.utils import generate_dataset

# Generate a small dataset for preview
data_vdp = generate_dataset(vdp, n_trajectories=10, t_end=8.0, dt=0.05, noise_std=0.05, seed=0)

print(f"Van der Pol dataset shape: x={data_vdp['x'].shape}, u={data_vdp['u'].shape}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].scatter(data_vdp['x'][:, 0], data_vdp['x'][:, 1], s=2, alpha=0.3, c=data_vdp['t'], cmap='viridis')
axes[0].set_xlabel('x₁'); axes[0].set_ylabel('x₂')
axes[0].set_title('State Distribution')

axes[1].scatter(data_vdp['x'][:, 0], data_vdp['dxdt'][:, 0], s=2, alpha=0.3, color='#2196F3')
axes[1].set_xlabel('x₁'); axes[1].set_ylabel('dx₁/dt')
axes[1].set_title('Derivative: dx₁/dt vs x₁')

axes[2].scatter(data_vdp['x'][:, 1], data_vdp['dxdt'][:, 1], s=2, alpha=0.3, color='#FF5722')
axes[2].set_xlabel('x₂'); axes[2].set_ylabel('dx₂/dt')
axes[2].set_title('Derivative: dx₂/dt vs x₂')

for ax in axes:
    ax.grid(alpha=0.2)

plt.suptitle('Training Data Preview — Van der Pol', fontweight='bold')
plt.tight_layout()
plt.show()

## Summary

| System | Key Challenge | Why PINN is Useful |
|--------|--------------|--------------------|
| Van der Pol | Limit cycle → origin | Embeds ẋ₁=x₂ exactly |
| Cart-Pole | Open-loop unstable, 4D | Captures nonlinear sine/cosine terms |
| CSTR | Multiple steady states, exponential nonlinearity | Data-efficient learning with physics priors |

**Next**: Notebook 02 — PINN Training and Validation